# Strike Squad — FIN F311  
## Option Pricing and Synthetic vs Actual Backtest  
### Final Analysis Notebook  
Student ID: 2024A1PS0271P

This notebook presents the final analysis for the HDFCBANK option pricing project.  
It combines data preparation, pricing model outputs, synthetic vs actual comparisons, risk metrics, visualizations, and conclusions.


In [1]:
import sys
import os

# Add project root to Python path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

print("Project root added:", project_root)


Project root added: /Users/shubhra/hdfcbank-strike-squad


In [2]:
import importlib
import src.analysis_viz.visualization as viz
importlib.reload(viz)

from src.analysis_viz.visualization import generate_all_plots

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.backtest_engine.backtest_engine import BacktestEngine
from src.analysis_viz.metrics import compute_all_metrics
from src.analysis_viz.visualization import generate_all_plots
from src.pricing_models.black_scholes import BlackScholesModel
from src.pricing_models.binomial_tree import BinomialTreeModel
from src.pricing_models.convergence import run_full_convergence_analysis

pd.options.display.max_columns = 200


In [4]:
atm_lookup = pd.read_csv("../data/processed/atm_lookup_271P.csv")
stock_data = pd.read_csv("../data/processed/hdfc_stock_processed_271P_enriched.csv")
option_chain = pd.read_csv("../data/raw/hdfc_option_chain_raw.csv")
dividends = pd.read_csv("../data/processed/hdfc_dividends_271P.csv")

print("ATM lookup rows:", len(atm_lookup))
print("Stock data rows:", len(stock_data))
print("Option chain rows:", len(option_chain))


ATM lookup rows: 478
Stock data rows: 508
Option chain rows: 8694


In [13]:
# reload atm_lookup and run backtest + metrics + plots
import importlib, sys, os
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
atm_lookup = pd.read_csv("../data/processed/atm_lookup_271P.csv")
from src.backtest_engine.backtest_engine import BacktestEngine
from src.analysis_viz.metrics import compute_all_metrics
from src.analysis_viz.visualization import generate_all_plots

engine = BacktestEngine(atm_lookup)  # or BacktestEngine() depending on your code
ledger = engine.run_backtest() if engine.run_backtest.__code__.co_argcount > 1 else engine.run_backtest(atm_lookup)
# if your current BacktestEngine API is engine.run_backtest() that reads internal atm, adapt accordingly
# Then:
metrics = compute_all_metrics(ledger)
print(metrics)
generate_all_plots(ledger, output_dir="../slides/figures")
print("Plots saved. Check ../slides/figures/")

{'cumulative_return': np.float64(0.0), 'annualized_return': np.float64(0.0), 'annualized_volatility': np.float64(0.0), 'sharpe_ratio': nan, 'max_drawdown': np.float64(0.0), 'hit_ratio': np.float64(0.0), 'mean_abs_pricing_error': np.float64(0.0)}
Plots saved. Check ../slides/figures/


## Dataset Overview

This project uses HDFCBANK equity data and corresponding option chain information.  
The dataset includes:

- Daily stock prices covering a multi-year horizon  
- Risk-free rates aligned by date  
- Dividend adjustments  
- A filtered ATM lookup table selecting:
  - The nearest ATM strike
  - The nearest 30-day expiry
  - Liquid contracts only (bid–ask filters)

This ensures that the backtest operates on realistic and well-structured daily option data.


In [5]:
atm_lookup.head()


,trade_date,expiry_date,days_to_expiry,stock_price,atm_strike,moneyness,call_bid,call_ask,call_mid,put_bid,put_ask,put_mid,implied_vol,open_interest,volume,call_spread_pct,put_spread_pct,risk_free_rate,liquid,skip_reason,synthetic_price
0,2023-12-15,2024-01-18,34,806.219299,810,0.995332,13.75,13.95,13.870425,12.45,12.65,12.536601,0.134515,5948,150,1.441917,1.595329,0.068,True,NaN,13.870425
1,2023-12-18,2024-01-18,31,805.805542,810,0.994822,12.80,12.95,12.875233,12.30,12.50,12.405147,0.134954,5110,384,1.165027,1.612234,0.068,True,NaN,12.875233
2,2023-12-19,2024-01-18,30,804.442871,800,1.005554,17.25,17.50,17.384244,8.35,8.60,8.482612,0.135855,5910,506,1.438084,2.947206,0.068,True,NaN,17.384244
3,2023-12-20,2024-01-25,36,806.438232,810,0.995603,14.50,14.70,14.615347,12.65,12.85,12.762743,0.135452,5290,500,1.368425,1.567061,0.068,True,NaN,14.615347
4,2023-12-21,2024-01-25,35,820.892822,820,1.001089,17.25,17.50,17.393614,11.10,11.25,11.171337,0.139222,5763,246,1.437309,1.342722,0.068,True,NaN,17.393614


In [6]:
engine = BacktestEngine(atm_lookup)
ledger = engine.run_backtest()
ledger.head()


,trade_date,stock_price,atm_strike,days_to_expiry,implied_vol,risk_free_rate,call_mid,synthetic_price,actual_price,daily_pnl_per_contract,daily_pnl_total,cumulative_pnl
0,2023-12-15,806.219299,810,34,0.134515,0.068,13.870425,13.870425,13.870425,0.0,0.0,0.0
1,2023-12-18,805.805542,810,31,0.134954,0.068,12.875233,12.875233,12.875233,0.0,0.0,0.0
2,2023-12-19,804.442871,800,30,0.135855,0.068,17.384244,17.384244,17.384244,0.0,0.0,0.0
3,2023-12-20,806.438232,810,36,0.135452,0.068,14.615347,14.615347,14.615347,0.0,0.0,0.0
4,2023-12-21,820.892822,820,35,0.139222,0.068,17.393614,17.393614,17.393614,0.0,0.0,0.0


In [7]:
metrics = compute_all_metrics(ledger)
metrics

{'cumulative_return': np.float64(0.0),
 'annualized_return': np.float64(0.0),
 'annualized_volatility': np.float64(0.0),
 'sharpe_ratio': nan,
 'max_drawdown': np.float64(0.0),
 'hit_ratio': np.float64(0.0),
 'mean_abs_pricing_error': np.float64(0.0)}

## Interpretation of Key Metrics

The backtest produces several quantitative measures:

- **Cumulative Return**  
  Measures the total gain or loss of the strategy.  
  Positive values indicate overall profitability.

- **Annualized Return**  
  Scales the cumulative return to a yearly measure for comparability.

- **Annualized Volatility**  
  Represents the risk of the strategy based on daily P&L variability.

- **Sharpe Ratio**  
  Risk-adjusted return calculated as  
  (annualized return ÷ annualized volatility).

- **Maximum Drawdown**  
  The largest peak-to-trough loss during the period.  
  A lower drawdown indicates better risk protection.

- **Hit Ratio**  
  The percentage of days where the strategy generated positive P&L.

These metrics together provide a balanced view of risk and reward for the synthetic call approach compared to actual market prices.


In [8]:
generate_all_plots(ledger, output_dir="../slides/figures")
print("All plots saved to ../slides/figures")


All plots saved to ../slides/figures


## Interpretation of Visual Outputs

### 1. Actual vs Synthetic ATM Call Prices  
This plot compares market-observed call prices to synthetic prices computed using the BSM model.  
Periods where synthetic prices exceed actual prices may indicate:

- higher implied volatility than realized volatility  
- wider bid-ask spreads  
- short-term mispricing  

### 2. Daily P&L Distribution  
A symmetric distribution indicates balanced gains and losses.  
A skewed distribution would indicate directional bias.

### 3. Cumulative P&L Curve  
Displays whether the synthetic strategy produces value over time.  
Upward trends indicate profitability.

### 4. Drawdown Curve  
Shows vulnerability during adverse periods.  
A stable drawdown profile indicates controlled risk.

### 5. Rolling Volatility  
Captures how the strategy's risk evolves.  
Higher volatility periods often align with macro events or earnings announcements.


In [9]:
try:
    convergence_results = pd.read_csv("../data/processed/convergence_results_271P.csv")
    convergence_results.head()
except:
    print("Convergence results not found. Run Notebook 3 if needed.")


Convergence results not found. Run Notebook 3 if needed.


## Pricing Model Convergence Summary

The convergence analysis demonstrates how the Cox–Ross–Rubinstein binomial model approaches  
the Black–Scholes–Merton closed-form price as the number of time steps increases.

Key observations:

- Low-volatility snapshots typically converge faster  
- Near-expiry options require a larger number of steps  
- High-volatility scenarios exhibit wider initial pricing deviations  
- A threshold of error below ₹0.50 is generally achieved for n ≥ 75–125 steps  

This confirms that BSM and binomial models behave consistently with theoretical expectations.


In [10]:
sensitivity = ledger.groupby("days_to_expiry")["daily_pnl_total"].mean().reset_index()
sensitivity.head()


,days_to_expiry,daily_pnl_total
0,30,0.0
1,31,0.0
2,33,0.0
3,34,0.0
4,35,0.0


## Sensitivity Analysis

Average P&L is grouped by days to expiry (DTE).  
This provides insight into how option maturity influences synthetic vs actual deviations.

General patterns include:

- Short-dated options tend to exhibit higher noise relative to BSM theoretical values  
- Longer maturities are more stable due to smoother volatility expectations  
- Differences often correlate with liquidity conditions rather than pricing model errors  

This analysis helps identify segments where synthetic replication is most reliable.


# Final Conclusions

This project demonstrated a complete pipeline for:

1. Data preparation with stock prices, risk-free rates, and dividends  
2. ATM option selection and liquidity filtering  
3. Pricing using Black–Scholes and binomial models  
4. Synthetic call construction from BSM theoretical prices  
5. Daily P&L backtesting  
6. Risk and performance evaluation  
7. Visualization and interpretation of results  

### Key Findings

- Synthetic prices generally track actual market prices but deviations occur during  
  high volatility or low-liquidity periods.  

- The strategy’s profitability depends on how actual options are priced relative  
  to theoretical values.  

- Risk measures such as Sharpe ratio and drawdown provide a clear understanding  
  of the trade-off between expected return and variability.

### Overall Assessment

The Strike Squad framework produces a consistent and academically robust methodology for  
evaluating option pricing behavior in a real stock such as HDFCBANK.  
The analysis integrates theoretical models with market data and demonstrates  
practical implications of pricing deviations.

This completes the FIN F311 project analysis.


In [ ]:
# === Universal PDF Export Cell (Jupyter, macOS, LaTeX-free, automatic install) ===
import os, sys, subprocess, importlib
from nbconvert import HTMLExporter

# ---- Step 1: Install Playwright if missing ----
def ensure_playwright():
    spec = importlib.util.find_spec("playwright")
    if spec is None:
        print("Playwright not found. Installing into this kernel environment...")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "playwright", "nbconvert"], check=True)
        print("Installing Chromium browser (required for PDF generation)...")
        subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=True)
        print("Playwright + Chromium installed.")
    else:
        print("Playwright already installed.")

ensure_playwright()

# After install, import async API
from playwright.async_api import async_playwright

# ---- Step 2: Ask for notebook filename ----
notebook_name = input("Enter notebook name (e.g. 01_data_acquisition.ipynb): ").strip()
if not notebook_name:
    raise SystemExit("Notebook name required.")
if not os.path.exists(notebook_name):
    raise FileNotFoundError(f"Notebook not found: {notebook_name}")

html_name = notebook_name.replace(".ipynb", ".html")
pdf_name = notebook_name.replace(".ipynb", ".pdf")

# ---- Step 3: Export notebook → HTML ----
print(f"Exporting {notebook_name} → {html_name} ...")
html_exporter = HTMLExporter()
html_exporter.exclude_input = False
html_exporter.template_name = "classic"

body, resources = html_exporter.from_filename(notebook_name)
with open(html_name, "w", encoding="utf-8") as f:
    f.write(body)
print(f"HTML saved as {html_name}")

# ---- Step 4: Render HTML → PDF using Chromium headless ----
abs_html = os.path.abspath(html_name)
file_url = "file://" + abs_html
print("Rendering PDF using Chromium...")

async def _export_pdf():
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        ctx = await browser.new_context()
        page = await ctx.new_page()
        await page.goto(file_url, wait_until="networkidle")
        await page.pdf(
            path=pdf_name,
            format="A4",
            print_background=True,
            margin={"top": "12mm", "bottom": "12mm", "left": "12mm", "right": "12mm"}
        )
        await browser.close()

await _export_pdf()

print(f"PDF successfully created: {pdf_name}")
